# ML-KEM-768 basemul CPA (비밀키 ŝ 전체 복구)

## 연구 목적과 분석 범위

이 노트북은 저장된 전력 트레이스로 ML-KEM-768 decapsulation 중 **A 지점인 비밀키 기반 `basemul` 연산**의 누출을 분석한다. 목표는 NTT 도메인 비밀 다항식 `ŝ`의 계수 후보마다 그 계수가 만드는 중간값과 측정 파형 사이의 Pearson 상관을 계산하고, 후보 순위와 누출 위치를 확인하는 것이다. 트레이스 수집은 별도 노트북 `mlkem-trace-basemul.ipynb`에서 수행하며, 여기서는 실측 `.npz`를 불러와 오프라인 분석한다.

ML-KEM-768의 파라미터는 `q=3329`, `N=256`, `K=3`이다. 따라서 `ŝ`는 세 개의 256계수 다항식으로 이루어져 총 768개 비밀 NTT 계수를 가진다. `P`는 한 다항식 안의 계수 쌍 번호이며, `a0=ŝ[VEC][2P]`, `a1=ŝ[VEC][2P+1]`로 쌍의 두 계수를 구분한다. 캡처마다 암호문과 파형이 행별로 대응하고, 분석 대상의 장기 비밀키는 고정되어 있다.

## 타깃 연산과 누출 가설

PQClean clean 구현의 `polyvec_basemul_acc_montgomery`는 암호문으로부터 복원한 공개 NTT 다항식 `b̂`와 비밀 다항식 `ŝ`를 쌍별로 곱하고 누산한다. 첫 `basemul` 쌍에서 조사하는 홀수 계수의 첫 곱셈은 개념적으로 다음과 같다.

\[
  r_0 = \operatorname{MontgomeryReduce}(a_1 \cdot b_1),\qquad
  a_1=\hat{s}[VEC][2P+1],\quad b_1=\hat{b}[VEC][2P+1].
\]

각 캡처의 `b1`은 해당 행의 암호문 `ct`에서 `u`를 압축 해제하고 NTT를 적용해 재현하므로 알려진 입력이다. `a1`은 복구 대상이다. 후보 `g∈{0,…,3328}`마다 `HW16(MontgomeryReduce(g·b1))`를 계산한다. HW16은 기기의 16-bit signed 결과를 16-bit 비트 패턴으로 해석한 Hamming weight이며, `b1`은 트레이스마다 다르므로 후보별 예측 누출도 트레이스마다 달라진다.

## CPA가 계산하는 값

CPA는 파형들을 먼저 평균내지 않는다. 후보 `g`마다 예측 누출 `L_g[t]`와 각 샘플 위치 `j`의 실측 전력 `W[t,j]`를 **같은 트레이스 인덱스 `t`끼리 짝지어** 상관을 계산한다.

\[
  \rho(g,j)=\operatorname{corr}_{t}(L_g[t],W[t,j]).
\]

선택한 시간 구간에서 후보별 점수를 구하고, 실제 키 계수의 순위를 사후에 계산한다. rank 0은 그 평가 조건에서 실제 계수가 가장 높은 점수를 받았음을 뜻한다. 이는 트레이스 평균 파형의 상관과 다르며, 서로 다른 암호문에서 발생한 계수별 누출 차이를 보존한다.

## 노트북 구성

1. PQClean clean의 정수 연산을 재현한다. Montgomery/Barrett reduction, NTT, 압축·복원 관련 포팅은 CPA 모델이 소프트웨어의 실제 중간값과 일치하도록 하기 위한 것이다.
2. 타깃 전용 `.npz`를 읽고 파형·암호문·키 배열의 정렬 및 필수 필드를 확인한다. TVLA 파일이나 B 캡처를 A CPA 입력으로 잘못 선택하지 않도록 파일 메타데이터도 검사한다.
3. 먼저 단일 계수 `a1`에서 후보별 모델·상관·순위를 계산하고 그래프로 확인한다. `WINDOW`는 분석할 샘플 구간이며, 전체 파형과 좁은 구간의 결과를 혼동하지 않도록 명시한다.
4. 트레이스 수를 늘리는 prefix 실험은 표본 수가 후보 순위에 미치는 영향을 보여준다. 이는 현재 캡처 조건에서의 수렴 진단이지, 최소 필요 트레이스 수의 증명은 아니다.
5. 전체 키 셀은 세 다항식의 768개 계수에 대해 모델 계산과 순위 검사를 반복하고, 결과를 조립·시각화한다.

## 결과를 해석할 때의 핵심 제한

현재 전체 계수 절차는 **완전한 blind 공격으로 해석하면 안 된다.** 각 계수의 후보 점수화는 `0..q−1` 전수 탐색으로 하지만, 부호 보정과 시간 위치/검색 밴드 선택 과정에서 저장된 정답 `ŝ`를 참조하는 코드가 있다. 따라서 rank 0 전부를 얻더라도 “정답을 모르는 공격자가 모든 샘플에서 독립적으로 키를 복구했다”는 의미는 아니다. 이 절차는 알려진 키로 POI·부호를 보정한 feasibility 또는 oracle-assisted 기준선이다. 진짜 blind 복구 주장을 위해서는 키와 무관한 POI·부호 선택, ghost 피크 구분 방법, 별도 검증 키/캡처가 필요하다.

결과는 현재 `.npz`의 펌웨어 구현, 최적화 수준, 클럭·샘플링, 트리거 위치, 정렬 상태에 한정된다. 데이터 수집 조건을 바꾼 배치를 섞어 순위를 개선했다고 간주하지 말고, 각 캡처의 메타데이터와 출처를 함께 기록한다. 정수 포팅 검증은 `mlkem-test.ipynb`와 함께 확인한다.

## 1. PQClean clean 산술 포팅: 모델이 기기 중간값을 재현하는 이유

CPA의 순위는 누출 가설뿐 아니라 그 가설에 들어가는 산술값에도 의존한다. 따라서 이 절의 Python 구현은 단순히 `mod q`에서 같은 값을 내는 것만으로 충분하지 않다. `int16_t` 변환, Montgomery reduction, Barrett reduction, NTT 순서 및 계수의 centered representative가 실제 PQClean clean 구현과 맞아야 한다. 같은 모듈러 잔여값이라도 16-bit 표현이 달라지면 Hamming weight가 달라질 수 있다.

아래 상수와 함수는 `firmware/mcu/pqm4/.../ml-kem-768/clean`의 연산 순서를 포팅한다. 암호문에서 계산한 `b̂=poly_ntt(Decompress(u))`와 `.npz`의 `dk`에서 파싱한 정답 `ŝ`를 독립 구현과 비교하는 테스트는 `mlkem-test.ipynb`에 둔다. 그 cross-check는 모델의 정합성을 뒷받침하지만, 그 자체로 전력 누출이나 공격 성능을 증명하지는 않는다.

### 실습 안내 — 이 셀은 “암호 수식 ↔ C 정수 연산” 번역기야

이 셀은 CPA 자체가 아니라, **암호문과 비밀키를 넣었을 때 보드가 계산할 값을 다시 만드는 도구**를 준비해. 아래 함수들을 역할별로 나눠 읽어 봐.

1. `to_i16`, `montgomery_reduce`, `fqmul`, `barrett_reduce`: C의 `int16_t` wrap과 q-기반 축약을 Python 정수에서도 재현해.
2. `ntt`: `u`를 NTT 영역으로 옮겨. `ZETAS` 순서와 butterfly 순서는 PQClean과 같아야 해.
3. `poly_frombytes`: decapsulation key의 12-bit 포맷을 256개 계수로 풀어.
4. `polyvec_decompress_vec`, `bhat_from_ct`: 암호문에서 CPA의 알려진 입력 `b̂`를 복원해.
5. `shat_from_dk`: 알려진 키를 읽어 평가용 정답 계수를 얻어. 이 값이 후보 점수에 쓰였는지, 사후 채점에만 쓰였는지를 이후 셀에서 구분할 거야.

**체크:** 같은 mod-q 값이어도 `int16` 비트 표현이 다르면 Hamming weight가 달라질 수 있어. 그래서 마지막 결과를 단순히 `% 3329`로 맞추는 것만으로는 누출 모델이 맞다고 할 수 없어.


In [ ]:
import numpy as np                              # 수치 계산

Q, QINV, N, K = 3329, -3327, 256, 3               # q=모듈러스, QINV=q^-1 mod 2^16, N=계수수, K=다항식수(768)
ZETAS = [                                         # NTT twiddle 상수 128개 (ntt.c와 동일해야 b̂ 재현됨)
    -1044, -758, -359, -1517, 1493, 1422, 287, 202, -171, 622, 1577, 182, 962, -1202,
    -1474, 1468, 573, -1325, 264, 383, -829, 1458, -1602, -130, -681, 1017, 732, 608,
    -1542, 411, -205, -1571, 1223, 652, -552, 1015, -1293, 1491, -282, -1544, 516, -8,
    -320, -666, -1618, -1162, 126, 1469, -853, -90, -271, 830, 107, -1421, -247, -951,
    -398, 961, -1508, -725, 448, -1065, 677, -1275, -1103, 430, 555, 843, -1251, 871,
    1550, 105, 422, 587, 177, -235, -291, -460, 1574, 1653, -246, 778, 1159, -147, -777,
    1483, -602, 1119, -1590, 644, -872, 349, 418, 329, -156, -75, 817, 1097, 603, 610,
    1322, -1285, -1465, 384, -1215, -136, 1218, -1335, -874, 220, -1187, -1659, -1185,
    -1530, -1278, 794, -1510, -854, -870, 478, -108, -308, 996, 991, 958, -1460, 1522, 1628,
]

def to_i16(x):                                      # 값을 int16 범위로 wrap (보드가 int16_t라 똑같이 잘라야 함)
    return ((int(x) + 0x8000) & 0xFFFF) - 0x8000  # +2^15 후 2^16 마스크 후 -2^15 = 부호있는 16비트

def montgomery_reduce(a):                         # a * R^-1 mod q (R=2^16) — reduce.c와 비트 단위 동일
    t = to_i16(to_i16(a) * QINV)                      # t = (int16_t)((int16_t)a * QINV)
    return (int(a) - t * Q) >> 16                 # (a - t*q) >> 16 (산술 시프트)

def fqmul(a, b):                                  # NTT 도메인 곱 = mont(a*b)
    return montgomery_reduce(to_i16(a) * to_i16(b))   # 두 int16을 곱해 몽고메리 리덕션

def barrett_reduce(a):                            # poly_reduce: ntt 뒤 보드가 적용하는 reduce (대표값 맞춤)
    v = ((1 << 26) + Q // 2) // Q                 # barrett 상수 round(2^26 / q)
    t = (v * to_i16(a) + (1 << 25)) >> 26           # t = round(a / q)
    return to_i16(to_i16(a) - to_i16(t * Q))            # a - t*q (centered 대표값)

def ntt(r):                                       # 정방향 NTT (normal -> bitreversed order)
    r = [to_i16(x) for x in r]                      # 입력을 int16로 (보드와 동일 표현)
    k, length = 1, 128                            # k=zeta 인덱스, length=butterfly 거리
    while length >= 2:                            # 길이 128,64,...,2 로 7단
        start = 0                                 # 각 단의 그룹 시작 위치
        while start < 256:                        # 256계수를 그룹 단위로 훑음
            zeta = ZETAS[k]; k += 1               # 이 그룹의 twiddle
            for j in range(start, start + length):# 그룹 내 butterfly
                t = fqmul(zeta, r[j + length])    # 짝 계수에 twiddle 곱
                r[j + length] = to_i16(r[j] - t)    # 아래쪽 = 위 - t
                r[j] = to_i16(r[j] + t)             # 위쪽 = 위 + t
            start = j + length + 1                # 다음 그룹 시작 (start + 2*length)
        length >>= 1                              # 다음 단은 거리 절반
    return r                                       # NTT 결과 리스트 반환

def poly_frombytes(a):                            # dk의 12비트 패킹을 NTT 비밀키 계수로 복원 (0..q-1)
    r = [0] * N                                   # 256계수 버퍼
    for i in range(N // 2):                       # 3바이트가 2계수
        r[2*i]   = ((a[3*i] >> 0) | (a[3*i+1] << 8)) & 0xFFF   # 하위 12비트 = 계수 0
        r[2*i+1] = ((a[3*i+1] >> 4) | (a[3*i+2] << 4)) & 0xFFF # 상위 12비트 = 계수 1
    return r                                       # ŝ 한 다항식

def polyvec_decompress_vec(a):                    # c1 압축(du=10) 해제: 320바이트 -> 256 normal 계수
    r = [0] * N; p = 0                            # r=출력, p=바이트 포인터
    for j in range(N // 4):                       # 5바이트가 4계수(10비트×4)
        t = ((a[p] | a[p+1] << 8),                # 10비트 값 4개 추출
             (a[p+1] >> 2 | a[p+2] << 6),
             (a[p+2] >> 4 | a[p+3] << 4),
             (a[p+3] >> 6 | a[p+4] << 2))
        p += 5                                    # 다음 5바이트로
        for kk in range(4):                       # 4계수 각각 Decompress
            r[4*j+kk] = ((t[kk] & 0x3FF) * Q + 512) >> 10   # round(q/2^10 * t)
    return r                                       # u 한 다항식 (normal 도메인)

def bhat_from_ct(ct):                             # 아는 값 b̂ = poly_ntt(Decompress(c1)) — 보드 poly_ntt와 동일
    out = []                                      # 다항식 K개
    for i in range(K):                            # 각 u 벡터 성분
        coeffs = polyvec_decompress_vec(ct[320*i:320*(i+1)])  # c1의 i번째 320바이트 -> u_i
        coeffs = ntt(coeffs)                      # 정방향 NTT (반환값 사용)
        out.append([barrett_reduce(x) for x in coeffs])       # ntt 뒤 poly_reduce 적용
    return out                                     # b̂ (NTT 도메인)

def shat_from_dk(dk):                             # 정답 비밀키 ŝ (채점용) — dk 앞 384*K 바이트
    return [poly_frombytes(dk[384*i:384*(i+1)]) for i in range(K)]  # 다항식 K개



## 2. basemul 캡처 `.npz` 불러오기: 파형과 입력의 행 정렬

`mlkem-trace-basemul.ipynb`가 저장한 `*-basemul-*.npz`를 우선 불러온다. 각 `traces[t]`는 같은 인덱스의 `ct[t]`와 연결되어야 CPA 모델이 올바르다. 분석 로더는 캡처 타깃과 필수 필드를 확인하고, TVLA의 `waves/labels` 파일이나 `poly_tomsg` 캡처가 섞이지 않도록 검사한다. 새 메타데이터가 없는 구형 파일은 명시적인 legacy fallback으로만 허용한다.

`TRACE_LIMIT`/`NT` 같은 설정이 있으면 실제로 분석한 prefix 크기를 결과와 함께 읽는다. `.npz`에서 비밀키를 가져오는 경우 해당 키는 known-key 채점 또는 보정에 쓰일 수 있으므로, 값 자체를 블로그 그림·출력에 공개하지 않는다.

In [ ]:
with np.load("../traces/mlkem768-clean-O3-basemul-20261030-042431.npz") as data:
    traces_full = data["traces"].astype(np.float32)
    cts_full = data["ct"]
    dk = data["dk"].tobytes()

traces, cts = traces_full, cts_full
PARAMS, IMPL, OPT = 768, "clean", "3"
T, S = traces.shape


## 3. 공격 타깃 설정: 계수 쌍에서 복구할 값 선택

`VEC`는 세 비밀 다항식 중 하나를, `P`는 그 다항식 안의 계수 쌍을 고른다. 이 단일계수 실험은 `a1=ŝ[VEC][2P+1]`을 후보화하며, 첫 번째 `basemul` 호출을 대상으로 할 때 `VEC=0`을 사용한다. 짝수 계수 `a0=ŝ[VEC][2P]`는 같은 곱셈 쌍의 다른 부분에 영향을 주므로 별도의 타깃이다.

`b1=NTT(u)[VEC][2P+1]`은 캡처별 암호문에서 재구성하는 값이다. 즉 CPA에서 알려진 입력은 각 캡처마다 달라지지만, 비밀계수 `a1`은 고정되어 있다. 이 구분이 후보별 누출 모델을 만드는 근거다.

In [3]:
VEC, P = 0, 7                                     # 타깃 좌표: a1 = ŝ[VEC][2P+1] (0<=P<128, A타깃은 VEC=0)

true_a1 = shat_from_dk(dk)[VEC][2*P + 1]          # 채점용 정답 계수 (rank 매길 때만 사용)
# b1[t] = b̂[VEC][2P+1] : 파형 t의 "아는 값"을 암호문에서 보드와 동일하게 재현
b1 = np.array([bhat_from_ct(bytes(ct))[VEC][2*P + 1] for ct in cts], dtype=np.int64)  # 파형마다 1개
print("true a1 =", true_a1, " | b1[:10] =", b1[:10])  # 정답과 b1 앞부분 확인

true a1 = 1203  | b1[:5] = [  727  1418  1523 -1006  -272  -712   870 -1319  -526  1081]


## 4. 단일계수 CPA: 후보 점수, 시간 창, ghost 피크

모든 `g=0..q−1` 후보에 대해 모델 `HW16(MontgomeryReduce(g·b1[t]))`를 만든다. 각 샘플 위치에서 트레이스 축으로 Pearson 상관을 계산하고, 점수와 rank를 구한다. 상관의 부호가 하드웨어 누출 방향과 반대일 수 있으므로 단일계수 feasibility에서는 절댓값을 쓰는 설정이 가능하지만, 전 구간의 최고점이 반드시 타깃 연산의 누출이라고 단정해서는 안 된다.

**A 타깃에서 특히 중요한 ghost 피크:** 한 `basemul` 계수쌍에는 `fqmul(a1,b1)`과 `fqmul(a0,b1)`이 모두 있고 두 연산이 서로 다른 시간에 실행된다. 따라서 `HW(Mont(g·b1))`라는 같은 형태의 모델이 a1뿐 아니라 a0 곱셈에도 반응할 수 있다. 전체 구간의 최고 상관이 더 강한 비타깃 연산을 가리키면 ghost가 된다. 파형과 상관 트레이스를 확인해 실제 a1 구간을 정하고, 좁힌 `WINDOW`에서 후보 순위를 다시 평가한다.

이 단일계수 창은 그래프/실험으로 정한 feasibility 조건이다. 창을 결과를 본 뒤 조정했다면 그 선택을 기록해야 하며, 그 rank를 독립적인 blind POI 선택 성능으로 설명하지 않는다.

### 실습 — CPA 후보 행렬을 읽기

코드는 `key_guesses`에 0부터 3328까지의 후보를 놓고, `guess_x_b1[g,t] = g × b1[t]`를 만든 다음 실제 C의 Montgomery reduction을 적용해. 그 결과의 16-bit 비트 수가 `hw_model[g,t]`야.

다음으로 트레이스 축(`t`)의 평균을 빼고, 각 후보와 각 샘플의 Pearson 상관을 계산해. **행은 후보, 열은 샘플**이라는 모양만 기억하면 행렬 곱의 의미가 보일 거야. `best_corr_per_guess`는 후보마다 창 안에서 가장 큰 절댓값 상관이고, `peak_sample_per_guess`는 그 위치야.

`WINDOW=(1900,1940)`은 `[1900,1940)` 샘플만 평가한다는 뜻이야. 먼저 넓은 파형을 보고 누출 창을 정한 뒤 이 구간을 고정하는 절차라, 이 rank는 이 선택 과정을 포함해 해석해야 해.

**생각해 보기:** 파형을 먼저 평균내면 각 암호문의 `b1[t]`와 어떤 후보 모델의 대응이 사라질까? 그래서 CPA는 파형을 평균내지 않고 행별로 짝지어 계산해.


In [4]:
WINDOW = (1900, 1940)  # isolates a1 from the a0 ghost peak

HW16_TABLE = np.array([bin(v).count("1") for v in range(1 << 16)], dtype=np.uint8)

def montgomery_reduce_vec(x):
    x  = x.astype(np.int64)
    lo = x.astype(np.int16)
    t  = (lo.astype(np.int32) * QINV).astype(np.int16)
    return ((x - t.astype(np.int64) * Q) >> 16).astype(np.int16)

key_guesses = np.arange(Q)
guess_x_b1  = key_guesses[:, None].astype(np.int64) * b1[None, :]
hw_model    = HW16_TABLE[montgomery_reduce_vec(guess_x_b1).astype(np.uint16)].astype(np.float64)
model_centered = hw_model - hw_model.mean(axis=1, keepdims=True)
model_norm     = np.sqrt((model_centered ** 2).sum(axis=1)) + 1e-12

lo, hi = (0, S) if WINDOW is None else WINDOW
win = traces[:, lo:hi].astype(np.float64)
win_centered = win - win.mean(axis=0)
win_norm     = np.sqrt((win_centered ** 2).sum(axis=0)) + 1e-12
corr = (model_centered @ win_centered) / (model_norm[:, None] * win_norm[None, :])

abs_corr              = np.abs(corr)
best_corr_per_guess   = abs_corr.max(axis=1)
peak_sample_per_guess = abs_corr.argmax(axis=1) + lo

ranked_guesses = np.argsort(best_corr_per_guess)[::-1]
true_rank      = int(np.where(ranked_guesses == true_a1)[0][0])
recovered_a1   = int(ranked_guesses[0])
print(f"window = {(lo, hi)}")
print(f"recovered a1 = {recovered_a1}   true a1 = {true_a1}   rank(0=best) = {true_rank}")
print("top5 (guess, score, peak_sample):")
for g in ranked_guesses[:5]:
    tag = "  <= TRUE" if g == true_a1 else ""
    print(f"  g={int(g):4d}  score={best_corr_per_guess[g]:.4f}  @sample {int(peak_sample_per_guess[g])}{tag}")

window = (1900, 1940)
recovered a1 = 1203   true a1 = 1203   rank(0=best) = 0
top5 (guess, score, peak_sample):
  g=1203  score=0.7520  @sample 1923  <= TRUE
  g=2126  score=0.6648  @sample 1923
  g=1044  score=0.3433  @sample 1909
  g=2285  score=0.3376  @sample 1909
  g=2435  score=0.2860  @sample 1938


## 5. 그래프 읽기: 점수 분포와 상관 트레이스

첫 그래프는 각 키 후보의 CPA 점수다. 실제 계수의 점이 가장 높은 후보와 분리되어 있는지, 2위 이하와 점수 간격이 얼마나 되는지 확인한다. rank 0만으로는 후보 간 여유나 선택 편향이 드러나지 않는다.

두 번째 그래프는 특정 모델의 샘플별 상관 크기다. 전체 구간을 보존해 a1 타깃 피크와 a0 ghost 피크를 비교한 뒤에야 분석 창을 정한다. 시각화가 전체 구간이고 순위 계산은 `WINDOW` 내부일 수 있으므로, 그림의 최대 피크와 실제 점수화 위치를 함께 보고한다.

### 그래프 실습 — 결과를 읽고 타깃을 구분하기

왼쪽은 `g`별 최대 점수야. 참값의 표시가 높은 것만 보지 말고, 1위와 다음 후보의 간격도 봐. 후보가 비슷한 점수를 가지면 rank 0이어도 안정적인 복구라는 뜻은 아니야.

오른쪽은 전체 샘플의 상관 트레이스야. 분석 창의 최대점이 더 큰 다른 피크(예: `a0` 연산의 ghost peak)와 같은 작업인지 확인해. 그래프에서 진짜 타깃에 해당하는 시점을 정한 근거를 말로 설명할 수 있어야 해.


In [5]:
from bokeh.plotting import figure, show                      # 인터랙티브 플롯
from bokeh.io import output_notebook                          # 노트북 출력
from bokeh.resources import INLINE                             # JS 인라인
from bokeh.layouts import gridplot                             # 두 플롯 가로 배치
from bokeh.models import BoxAnnotation, Span, HoverTool              # 윈도우 음영, 호버

output_notebook(INLINE)                                         # 출력 모드 켜기

full_centered = traces.astype(np.float64) - traces.astype(np.float64).mean(axis=0)  # 전체 파형 평균 제거
full_norm     = np.sqrt((full_centered ** 2).sum(axis=0)) + 1e-12   # 샘플별 norm
corr_true = np.abs((model_centered[true_a1]      @ full_centered) / (model_norm[true_a1]      * full_norm))  # 정답 추측 상관트레이스
corr_best = np.abs((model_centered[recovered_a1] @ full_centered) / (model_norm[recovered_a1] * full_norm))  # 1등 추측 상관트레이스

# 왼쪽: 추측별 CPA 점수 (상관 vs 추측값)
p1 = figure(width=760, height=380, title="CPA score per guess",
            x_axis_label="key guess", y_axis_label="max |corr|")
p1.line(key_guesses, best_corr_per_guess, line_color="gray", line_width=1, legend_label="all guesses")
p1.scatter([true_a1], [best_corr_per_guess[true_a1]], size=9, color="red", legend_label=f"true a1={true_a1}")
p1.add_tools(HoverTool(tooltips=[("guess", "$x{0}"), ("|corr|", "$y{0.0000}")]))
p1.legend.click_policy = "hide"                                 # 범례 클릭 = 숨기기/보이기

# 오른쪽: 상관 트레이스 (전체 구간) — a1/a0 ghost 피크 위치 비교
x = np.arange(len(corr_true))
p2 = figure(width=760, height=380, title="correlation trace (full)",
            x_axis_label="sample", y_axis_label="|corr|")
p2.line(x, corr_true, line_color="navy",   line_width=1, legend_label=f"true a1={true_a1}")
p2.line(x, corr_best, line_color="orange", line_width=1, alpha=0.7, legend_label=f"best g={recovered_a1}")
if WINDOW is not None:                                          # 윈도우 지정 시 음영 박스
    p2.add_layout(BoxAnnotation(left=WINDOW[0], right=WINDOW[1], fill_color="orange", fill_alpha=0.15))
p2.add_tools(HoverTool(tooltips=[("sample", "$x{0}"), ("|corr|", "$y{0.0000}")]))
p2.legend.click_policy = "hide"

show(gridplot([[p1, p2]]))                                      # 가로 배치로 표시

Loading BokehJS ...

## 6. 트레이스 수별 수렴: 무엇을 보여주는가

앞쪽 `n`개 트레이스만 사용해 같은 모델과 같은 선택 창으로 후보 순위를 반복한다. 순위가 trace prefix에 따라 어떻게 변하는지 보면 현재 캡처에서 표본 수가 CPA 구분력에 미치는 영향을 확인할 수 있다.

표본 수 목록에서 rank 0이 처음 관측된 값은 **시험한 prefix 중 첫 성공값**일 뿐, 최소 복구 트레이스 수가 아니다. 정답 피크·윈도우를 고정했다면 이 진단도 known-key 보정 조건을 계승한다. 결과에는 prefix 목록, 창, 캡처 배치 및 정답 기준 사용 여부를 같이 적는다.

In [6]:


steps = [i for i in range(2, 21)]  # 늘려볼 트레이스 수
rank_curve = []                                   # 각 n의 rank 기록

for n in steps:                                   # 트레이스 n개만으로 반복
    model_n          = hw_model[:, :n]            # 앞 n개 트레이스의 모델
    model_n_centered = model_n - model_n.mean(axis=1, keepdims=True)       # 평균 제거
    model_n_norm     = np.sqrt((model_n_centered ** 2).sum(axis=1)) + 1e-12  # norm

    win_n          = traces[:n, lo:hi].astype(np.float64)                  # 앞 n개 × 셀4 WINDOW 구간
    win_n_centered = win_n - win_n.mean(axis=0)                            # 평균 제거
    win_n_norm     = np.sqrt((win_n_centered ** 2).sum(axis=0)) + 1e-12    # norm

    corr_n  = (model_n_centered @ win_n_centered) / (model_n_norm[:, None] * win_n_norm[None, :])  # 상관
    score_n = np.abs(corr_n).max(axis=1)          # 추측별 점수
    rank_n  = int((score_n > score_n[true_a1]).sum())  # 정답보다 높은 추측 수(0=복구)
    rank_curve.append(rank_n)                     # 기록
    print(f"N={n:6d}  rank={rank_n}")             # 진행 표시

p = figure(width=800, height=400, title=f"CPA convergence (VEC={VEC}, P={P}, window={(lo, hi)})",
           x_axis_label="traces", y_axis_label="rank of true coeff (0=recovered)")
p.line(steps, rank_curve, line_color="navy", line_width=1.5)
p.scatter(steps, rank_curve, size=5, color="navy")
p.add_layout(Span(location=0, dimension="width", line_color="red", line_dash="dashed"))  # rank 0 기준선
p.add_tools(HoverTool(tooltips=[("traces", "$x{0}"), ("rank", "$y{0}")]))
show(p)

Loading BokehJS ...

N=     2  rank=585
N=     3  rank=251
N=     4  rank=1199
N=     5  rank=815
N=     6  rank=281
N=     7  rank=63
N=     8  rank=54
N=     9  rank=13
N=    10  rank=6
N=    11  rank=4
N=    12  rank=55
N=    13  rank=35
N=    14  rank=23
N=    15  rank=16
N=    16  rank=8
N=    17  rank=6
N=    18  rank=3
N=    19  rank=3
N=    20  rank=2


## 7. 전체 비밀키 계수 스윕: 실행 절차와 oracle 경계

이 셀은 `K=3`개 다항식의 256계수, 즉 `a0=ŝ[VEC][2P]`와 `a1=ŝ[VEC][2P+1]`을 합쳐 768개 계수를 각각 후보화한다. 각 계수에서 암호문으로부터 트레이스별 `b`를 재현하고 `q=3329` 후보의 Montgomery/HW 모델을 계산한다. 결과에는 추정 후보, 참값, rank, 누출 점수 및 선택된 샘플을 담아 전체 `ŝ` 행렬로 조립한다.

**현재 자동화의 핵심 보정:** 부호 다수결은 알려진 키의 누출 상관으로 보정되며, 계수별 피크·검색 밴드에도 `shat_true`가 참조된다. 그런 다음 해당 위치 주변에서 후보 `g`를 순위화한다. 즉 “후보를 전수 비교한다”는 말은 맞지만, 전체 절차의 시간 위치와 부호가 blind로 발견됐다는 뜻은 아니다. 현 결과는 정답 보정이 들어간 전체계수 feasibility 평가이며, 독립 키에서 검증하기 전에는 완전 blind 전체 키 복구로 주장하지 않는다.

실제 블로그 보고에는 총 계수 수, 신호 약한 좌표 수, rank별 성공 수, 실제 키 일치율, trace count, 샘플 창, 부호·피크 선택의 정답 의존성을 함께 기록한다. 셀은 진행 상황을 주기적으로 출력하므로 장시간 실행 중 어느 계수까지 처리됐는지도 확인할 수 있다.

### 실습 — 768계수 스윕을 네 단계로 따라가기

이 셀은 긴 코드를 한 번에 읽지 말고, 다음 네 부분으로 끊어서 봐.

1. **데이터/표현 준비:** `NT`개 파형과 암호문을 맞춰 자르고, 각 행의 `b̂`와 정답 `ŝ`를 만든다.
2. **누출 방향 보정:** 첫 다항식의 앞쪽 계수들에서 참값 상관 부호를 모아 `LEAK_SIGN`을 정한다. 이 단계부터 정답 키를 쓰므로 현재 방식은 blind가 아니야.
3. **계수별 위치 찾기와 후보 평가:** `VEC=0`에서 샘플-계수 위치 관계를 구하고, `VEC=1,2`는 앞 블록의 피크와 추정 간격으로 검색 밴드를 정한다. `model_for`는 계수의 짝/홀에 따라 `b̂` 인덱스를 선택하고, `cpa_rank`는 피크 주변 작은 창에서 후보를 비교한다.
4. **요약:** 점수가 약한 좌표는 `rank=-1`; 나머지는 실제 계수의 rank와 추정값을 기록한다.

**중요:** 전체 키를 후보로 돌린다는 사실만으로 blind 공격은 아니야. 참값으로 부호·피크·검색 밴드를 정하는 코드는 공격자의 사전 지식에 해당해. 이 셀은 정확히는 known-key/oracle 기준선이야.


In [ ]:
import time

NT = min(1000, len(traces_full))
traces = traces_full[:NT]
cts = cts_full[:NT]
print(f"using {NT} traces")

# 전체 파형 평균 제거(부호 유지 — abs 아님). 피크 탐색/부호 캘리브/랭킹에 공통 사용
trace_values = traces.astype(np.float64)
full_centered = trace_values - trace_values.mean(axis=0)
full_norm     = np.sqrt((full_centered ** 2).sum(axis=0)) + 1e-12

bhats = np.array([bhat_from_ct(bytes(ct)) for ct in cts])
shat_true = np.array(shat_from_dk(dk))
print("bhats:", bhats.shape, " shat_true:", shat_true.shape)

HALF_WIDTH = 3
SCORE_WEAK = 0.2

def model_for(VEC_, c):
    # '첫 fqmul = 깨끗한 레지스터 쓰기'인 강한 누출을 겨냥:
    #   홀수 c = a1 -> fqmul(a1, b1),  b1 = b̂[c]      (r[0] 첫 연산, |corr|~0.86)
    #   짝수 c = a0 -> fqmul(a0, b1),  b1 = b̂[c+1]    (r[1] 첫 연산, |corr|~0.94)  ← 약한 a0·b0(0.57) 대신
    b_idx = c if (c % 2 == 1) else c + 1
    b  = bhats[:, VEC_, b_idx].astype(np.int64)
    tv = int(shat_true[VEC_, c])
    gxb  = key_guesses[:, None].astype(np.int64) * b[None, :]
    hw   = HW16_TABLE[montgomery_reduce_vec(gxb).astype(np.uint16)].astype(np.float64)
    hw_c = hw - hw.mean(axis=1, keepdims=True)
    hw_n = np.sqrt((hw_c ** 2).sum(axis=1)) + 1e-12
    return hw_c, hw_n, tv

def corr_true(hw_c, hw_n, tv, lo_s=0, hi_s=None):
    hi_s = S if hi_s is None else hi_s
    seg  = full_centered[:, lo_s:hi_s]
    return lo_s, (hw_c[tv] @ seg) / (hw_n[tv] * full_norm[lo_s:hi_s])

def cpa_rank(hw_c, hw_n, tv, peak, sign, half=HALF_WIDTH):
    lo_c, hi_c = max(0, peak - half), min(S, peak + half)
    win   = traces[:, lo_c:hi_c].astype(np.float64)
    win_c = win - win.mean(axis=0)
    win_n = np.sqrt((win_c ** 2).sum(axis=0)) + 1e-12
    corr  = (hw_c @ win_c) / (hw_n[:, None] * win_n[None, :])
    pk    = min(peak - lo_c, corr.shape[1] - 1)
    score = sign * corr[:, pk]
    ranked = np.argsort(score)[::-1]
    return int(ranked[0]), int(np.where(ranked == tv)[0][0]), float(np.abs(corr).max())


results = []
total = K * N
t0 = time.time()

# ── 누출 부호 캘리브(핵심) ──────────────────────────────────────────────
# HW가 전력과 +/- 어느 쪽으로 상관되는지는 기기 특성(전 계수 공통 상수).
# 누출은 쌍극(+lobe/-lobe)이라 한 계수만 보면 노이즈로 부호가 뒤집힘
#  -> VEC0 앞쪽 센 계수들의 (argmax|corr| 지점 부호)를 |corr|로 가중 '다수결' 해서 안정적으로 결정
sign_vote = 0.0
for c in range(min(20, N)):
    hw_c, hw_n, tv = model_for(0, c)
    _, cr = corr_true(hw_c, hw_n, tv)
    k = int(np.argmax(np.abs(cr)))
    sign_vote += np.sign(cr[k]) * abs(cr[k])
LEAK_SIGN = float(np.sign(sign_vote)) or 1.0
print(f"leak sign calibrated (weighted majority): {LEAK_SIGN:+.0f}")

def find_peak(hw_c, hw_n, tv, lo_s=0, hi_s=None):
    off, cr = corr_true(hw_c, hw_n, tv, lo_s, hi_s)
    k = int(np.argmax(LEAK_SIGN * cr))
    return off + k

# 1) VEC=0: 신호 뚜렷 → 전체 S에서 피크 탐색 + '계수 c ↔ 샘플' 선형관계 캘리브
vec0_peaks = []
for c in range(N):
    hw_c, hw_n, tv = model_for(0, c)
    peak = find_peak(hw_c, hw_n, tv)
    recovered, rank, mag = cpa_rank(hw_c, hw_n, tv, peak, LEAK_SIGN)
    if mag < SCORE_WEAK:
        rank = -1
    results.append((0, c, recovered, tv, rank, peak, mag))
    vec0_peaks.append(peak)
    if len(results) % 64 == 0:
        print(f"{len(results):4d}/{total}  VEC=0 c={c:3d} rec={recovered:4d} true={tv:4d} rank={rank} |corr|={mag:.3f}  ({time.time()-t0:.1f}s)")

slope, intercept = np.polyfit(np.arange(N), vec0_peaks, 1)
print(f"VEC=0 calibration: slope={slope:.2f} samples/coeff, intercept={intercept:.1f}")

# 2) VEC=1,2: 한 VEC 블록 길이(slope·N)로 '밴드'를 잡고 그 밴드 전체에서 탐색.
#    밴드가 넓어도 각 계수 모델은 자기 b1에만 강하게 반응 → 엉뚱한 계수/블록에 안 걸리고,
#    블록 사이 poly_add 위치를 몰라도 됨(GAP 추정 불필요). 랭킹은 아래 cpa_rank의 좁은(±HALF_WIDTH) 윈도우가 담당.
block_len = slope * N
prev_end  = max(vec0_peaks)
for VEC_ in (1, 2):
    band_lo = max(0, int(prev_end - slope * 3))
    band_hi = min(S, int(prev_end + block_len * 1.3))
    vec_peaks, vec_mags = [], []
    for c in range(N):
        hw_c, hw_n, tv = model_for(VEC_, c)
        peak = find_peak(hw_c, hw_n, tv, band_lo, band_hi)
        recovered, rank, mag = cpa_rank(hw_c, hw_n, tv, peak, LEAK_SIGN)
        if mag < SCORE_WEAK:
            rank = -1
        results.append((VEC_, c, recovered, tv, rank, peak, mag))
        vec_peaks.append(peak); vec_mags.append(mag)
        if len(results) % 64 == 0:
            print(f"{len(results):4d}/{total}  VEC={VEC_} c={c:3d} rec={recovered:4d} true={tv:4d} rank={rank} |corr|={mag:.3f}  ({time.time()-t0:.1f}s)")
    prev_end = max(vec_peaks)
    print(f"  VEC={VEC_} band=({band_lo},{band_hi})  mean|corr|={np.mean(vec_mags):.3f}")

ranks = np.array([r[4] for r in results])
ok   = ranks == 0
weak = ranks == -1
print()
print(f"전체 {total}계수 중 복구 성공(rank=0): {int(ok.sum())} / {total}  ({100*ok.mean():.2f}%)")
print(f"신호 없음(캡처 부족 추정, |corr|<{SCORE_WEAK}): {int(weak.sum())}개")
if (~ok & ~weak).any():
    fails = [(r[0], r[1], r[2], r[3], r[4], round(r[6], 3)) for r in results if r[4] not in (0, -1)]
    print("신호는 있는데 복구 실패 (VEC, c, recovered, true, rank, |corr|):")
    for fr in fails[:20]:
        print("  ", fr)

using 1000 / 1000 traces
bhats: (1000, 3, 256)  shat_true: (3, 256)
leak sign calibrated (weighted majority): -1
  64/768  VEC=0 c= 63 rec=3209 true=3209 rank=0 |corr|=0.395  (13.8s)
 128/768  VEC=0 c=127 rec= 551 true= 551 rank=0 |corr|=0.404  (25.4s)
 192/768  VEC=0 c=191 rec=2543 true=2543 rank=0 |corr|=0.346  (36.7s)
 256/768  VEC=0 c=255 rec=1435 true=1435 rank=0 |corr|=0.502  (47.2s)
VEC=0 calibration: slope=126.99 samples/coeff, intercept=137.7
 320/768  VEC=1 c= 63 rec=3258 true=3258 rank=0 |corr|=0.500  (57.4s)
 384/768  VEC=1 c=127 rec=3290 true=3290 rank=0 |corr|=0.420  (66.5s)
 448/768  VEC=1 c=191 rec= 922 true= 922 rank=0 |corr|=0.404  (75.0s)
 512/768  VEC=1 c=255 rec=1030 true=1030 rank=0 |corr|=0.509  (84.8s)
  VEC=1 band=(32123,74765)  mean|corr|=0.657
 576/768  VEC=2 c= 63 rec=2948 true=2948 rank=0 |corr|=0.473  (94.1s)
 640/768  VEC=2 c=127 rec=1963 true=1963 rank=0 |corr|=0.357  (102.7s)


### 복구된 ŝ 조립 및 검증

이 절은 계수별 결과를 `(K,N)` 행렬 `recovered_shat`으로 모으고 실제 `shat_true`와 좌표별 비교한다. `rank=-1`은 CPA 점수가 임계값 미만인 신호 미확인 좌표로 따로 표시한다. 전체 일치율과 신호가 확인된 좌표에서의 일치율은 서로 다른 지표이므로 함께 확인한다.

`match.all()`은 현재 저장 키를 기준으로 한 사후 비교다. 앞 단계의 `shat_true` 사용 여부, 시간 위치·부호 보정, 캡처 배치의 조건이 그대로 따라오므로 이 값만으로 blind 공격 주장을 하지 않는다. 저장되는 결과 `.npz`는 분석 산출물이며 원본 측정 캡처와 구분한다.

In [8]:
recovered_shat = np.zeros((K, N), dtype=np.int64)
weak_mask = np.zeros((K, N), dtype=bool)
for VEC_, c, recovered, tv, rank, peak, score in results:
    recovered_shat[VEC_, c] = recovered
    weak_mask[VEC_, c] = (rank == -1)

match = (recovered_shat == shat_true)
print("계수 단위 일치율(전체):", f"{match.mean()*100:.2f}%", f"({int(match.sum())}/{match.size})")
print("신호 있는 계수 중 일치율:", f"{match[~weak_mask].mean()*100:.2f}%", f"({int(match[~weak_mask].sum())}/{int((~weak_mask).sum())})")
print("전체 키 완전 일치:", bool(match.all()))

if match.all():
    print("[OK] s-hat 768계수 전부 CPA로 복구 성공 -- A측(비밀키 basemul) 전체 키 복구 완료")
elif weak_mask.any():
    print(f"[PARTIAL] 신호 없음(캡처 부족 추정) 좌표 {int(weak_mask.sum())}개 -> SAMPLES 늘려서 재캡처 필요")
    bad = np.argwhere(~match & ~weak_mask)
    print("[FAIL] 신호는 있는데 틀린 좌표 (VEC, c):", bad.tolist()[:20])
else:
    bad = np.argwhere(~match)
    print("[FAIL] 불일치 좌표 (VEC, c):", bad.tolist()[:20])

import pathlib
stamp = time.strftime("%Y%m%d-%H%M%S")
out_path = pathlib.Path("../traces") / f"mlkem{PARAMS}-{IMPL}-O{OPT}-cpa-basemul-recovered_shat-{stamp}.npz"
np.savez_compressed(out_path, recovered_shat=recovered_shat, shat_true=shat_true, match=match, weak_mask=weak_mask)
print("saved ->", out_path)

계수 단위 일치율(전체): 100.00% (768/768)
신호 있는 계수 중 일치율: 100.00% (768/768)
전체 키 완전 일치: True
[OK] s-hat 768계수 전부 CPA로 복구 성공 -- A측(비밀키 basemul) 전체 키 복구 완료
saved -> ..\traces\mlkem768-clean-O3-cpa-basemul-recovered_shat-20261102-182927.npz


### 시각화 — 계수별 복구 결과

Bokeh 그림은 복구된 계수와 참값, rank/score 또는 신호 약한 위치를 훑어 패턴을 찾기 위한 진단 자료다. 특정 계수의 실패가 다항식 전체에 몰리는지, `a0/a1` 또는 `VEC`별로 달라지는지 살펴볼 수 있다.

그림의 각 점은 위 CPA 셀에서 계산한 결과를 표현할 뿐 독립적인 추가 검증은 아니다. 캡션에는 실측/재분석 상태, 사용한 트레이스 수와 oracle 보정 여부를 명시한다.

In [9]:


idxs = np.arange(len(results))                                # 0..767 (VEC*256 + c 순서)
weak = ranks == -1                                             # 신호 없음(캡처 부족 추정)
fail = (~ok) & (~weak)                                         # 신호는 있는데 틀린 경우
p = figure(width=1000, height=350,
           title=f"전체 비밀키 계수별 CPA 복구 결과 (성공 {int(ok.sum())}/{total}, 신호없음 {int(weak.sum())}, 실패 {int(fail.sum())})",
           x_axis_label="coefficient index (VEC*256 + c)", y_axis_label="rank of true value (0=success, -1=신호없음)")
p.scatter(idxs[ok],   ranks[ok],   size=3, color="green",  legend_label="success")
p.scatter(idxs[fail], ranks[fail], size=6, color="red",    legend_label="fail")
p.scatter(idxs[weak], ranks[weak], size=6, color="orange", legend_label="no signal (capture too short)")
p.add_tools(HoverTool(tooltips=[("idx", "$x{0}"), ("rank", "$y{0}")]))
p.legend.click_policy = "hide"
show(p)

Loading BokehJS ...